In [14]:
from __future__ import annotations
import torch
import torch.nn as nn

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import sys
import os
%matplotlib inline

# Question 1:
Implement and train an MLP as
specified in the assignment using PyTorch and multi-class cross entropy as the cost function. Experiment with
each of the four datasets to find the best number of nodes k from {2, 3, 5, 7, 9} in the hidden layer.

### Define network + dataloader

In [15]:
from dataclasses import dataclass
from typing import Any, Literal
import pathlib

from torch.utils.data import Dataset, DataLoader
import pandas as pd

# NETWORK DEFINITION
class ConorNet(nn.Module):
    """
    1xk neural network for HW1

    Here is a good clear reference for good nn code:
    https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html
    """

    @dataclass(eq=True, order=True, unsafe_hash=True)
    class Hyperparams:
        """Hyperparameters for training + model"""
        k: int = 7
        """Number of nodes in the hidden layer. One of {2, 3, 5, 7, 9}"""

        activation: str | Literal["tanh", "relu", "sigmoid", "leaky"] = "tanh"
        learning_rate: float = 1E-3
        training_epochs: int = 5
        batch_size: int = 50

    def __init__(self, p: Hyperparams):
        super().__init__()
        match p.activation:
            case "tanh":
                self._act = nn.Tanh()
            case "relu":
                self._act = nn.ReLU()
            case "sigmoid":
                self._act = nn.Sigmoid()
            case "leaky":
                self._act = nn.LeakyReLU()
            case _:
                raise NotImplementedError("Unrecognized activation function.")

        self.hparams = p

        # our network performs binary classification on an R2 input space, predicting 0 or 1.
        self._net = nn.Sequential(
            nn.Linear(2, self.hparams.k, bias=True),
            self._act,
            nn.Linear(self.hparams.k, 2, bias=True)
        )

    def forward(self, x):
        logits = self._net(x)
        # we need to apply sigmoid+thresh or softmax on these outputs to get actual output labels
        # but let's leave that to the calling code.
        return logits
    
# DATALOADERS
class HW1Dataset(Dataset):
    """Dataset representation for the given CSV files

    they all have the same columns etc.
    """
    def __init__(self, csv_file: pathlib.Path) -> None:
        self.data = pd.read_csv(csv_file, header=0)

        # sanity check
        if list(self.data.columns) != ["label", "x1", "x2"]:
            raise ValueError(f"Incorrect column names {self.data.columns}")

        self.X = self.data[["x1", "x2"]].to_numpy(dtype=np.float32)
        self.y = self.data["label"].to_numpy(dtype=np.int64)
    
    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(self, index) -> tuple[torch.Tensor, torch.Tensor]:
        return torch.tensor(self.X[index]), torch.tensor(self.y[index])

DSET_ROOT = pathlib.Path.cwd()
"""
Dataset root directory.

Sticking with the convention in the assignment zip, which is a flat directory with data + code.
"""

def smoke_test() -> None:
    # perform an initial evaluation, just to smoke test the code above.
    # doing this in function scope to not pollute the global namespace
    # i hate messy jupyter notebooks lol
    dset_path = DSET_ROOT / "spiral_train.csv"
    p = ConorNet.Hyperparams()

    train_dset = HW1Dataset(dset_path)
    train_dloader = DataLoader(train_dset, batch_size=p.batch_size, shuffle=True)

    print(f"Dataset: {dset_path}")
    print(f"len={len(train_dloader)}")
    xbatch, ybatch = next(iter(train_dloader))
    print(f"feature batch shape: {xbatch.size()}")
    print(f"label batch shape: {ybatch.size()}")
    print(f"idx 0--feat: {xbatch[0]}, label: {ybatch[0]}")

    print("\nEvaluating model...")
    model = ConorNet(p)
    # eval the model on the first batch we get
    logits: torch.Tensor = model(xbatch)
    logits = nn.Softmax(dim=1)(logits)
    print("logits: ")
    print(logits)

    # take argmax to get our yhat (since output node 0 = 0, output node 1 = 1)
    yhat = logits.argmax(dim=1)
    print("yhat:")
    print(yhat)

smoke_test()

Dataset: /home/conor/Documents/W2026/demeter_msai437/hw1/spiral_train.csv
len=4
feature batch shape: torch.Size([50, 2])
label batch shape: torch.Size([50])
idx 0--feat: tensor([  8.6949, -11.4961]), label: 0

Evaluating model...
logits: 
tensor([[0.5399, 0.4601],
        [0.5414, 0.4586],
        [0.3691, 0.6309],
        [0.5892, 0.4108],
        [0.3888, 0.6112],
        [0.3914, 0.6086],
        [0.3895, 0.6105],
        [0.4024, 0.5976],
        [0.5507, 0.4493],
        [0.5063, 0.4937],
        [0.5888, 0.4112],
        [0.3997, 0.6003],
        [0.5543, 0.4457],
        [0.5400, 0.4600],
        [0.3923, 0.6077],
        [0.4203, 0.5797],
        [0.3464, 0.6536],
        [0.5451, 0.4549],
        [0.5893, 0.4107],
        [0.5510, 0.4490],
        [0.4014, 0.5986],
        [0.5400, 0.4600],
        [0.3508, 0.6492],
        [0.4017, 0.5983],
        [0.3990, 0.6010],
        [0.5393, 0.4607],
        [0.5586, 0.4414],
        [0.5867, 0.4133],
        [0.3994, 0.6006],
       

### Create Testing & Plotting Functions
Let's make my life easier by mostly automating:
- hyperparameter search (exhaustively over k, hand-wavily over other params)
- helpful debug plots for the hyperparam search
- plots & reports for the best set of hparams as required in the assignment

My general strategy will be to try to optimize the hyperparams other than k at k=5 for the two gaussians test, 
because I have a vague sense that'll be a pretty decent value with enough training epochs.

Then, once I have sane values for the other params, I'll try out all k values.

In [16]:
# first, some one-time setup stuff

# load all the datasets into a convenient map
def load_datasets() -> dict[str, dict[str, HW1Dataset]]:
    """Load all the datasets into a convenient map."""
    names = ["center_surround", "spiral", "two_gaussians", "xor"]
    types = ["test", "train", "valid"]

    dsets = dict()
    count = 0
    for name in names:
        dsets[name] = dict()
        for t_ in types:
            path = DSET_ROOT / f"{name}_{t_}.csv"
            dsets[name][t_] = HW1Dataset(path)
            count += 1
    
    print(f"loaded {count} datasets.")
    return dsets

DSETS = load_datasets()

loaded 12 datasets.


In [17]:
from itertools import product

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def iterate_hyperparams_except_k():
    """Iterator for all the hyperparam combos i want to try"""
    k = [7]
    acts = ["tanh", "relu", "sigmoid", "leaky"]
    learns = [0.1, 0.01, 0.001]
    epochs = [5, 10, 20]
    batch = [1, 5, 50, 100]

    for k, act, lr, ep, bs in product(k, acts, learns, epochs, batch):
        yield ConorNet.Hyperparams(k=k, activation=act, learning_rate=lr, training_epochs=ep, batch_size=bs)

def train_all_but_k() -> dict[ConorNet.Hyperparams, ConorNet]:
    """
    Train a model for each of the hyperparam combos

    Returns dict of hparams -> model
    """

    dset_name = "two_gaussians"
    dset = DSETS[dset_name]
    out = dict()
    for hparam in iterate_hyperparams_except_k():
        print(f"\n~~~~~~~~~~~~~~~~~\nTRIAL: {hparam}")
        print(f"Dataset: {dset_name}; Device: {DEVICE}")

        # train the model
        loader = DataLoader(dset["train"], batch_size=hparam.batch_size, shuffle=True)

        model = ConorNet(hparam)
        # could maybe have the optimizer as a hyperparam later?
        optimizer = torch.optim.Adam(model.parameters(), lr=hparam.learning_rate)
        loss_fn = nn.CrossEntropyLoss()
        model.to(DEVICE)
        
        for epoch in range(hparam.training_epochs):
            print(f"EPOCH {epoch}")
            for xbatch, ybatch in loader:
                xbatch = xbatch.to(DEVICE)
                ybatch = ybatch.to(DEVICE)
                logits = model(xbatch)
                loss = loss_fn(logits, ybatch)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step() 
        
        # model is trained
        out[hparam] = model
    
    return out

models = train_all_but_k() 


~~~~~~~~~~~~~~~~~
TRIAL: ConorNet.Hyperparams(k=7, activation='tanh', learning_rate=0.1, training_epochs=5, batch_size=1)
Dataset: two_gaussians; Device: cuda
EPOCH 0
EPOCH 1
EPOCH 2
EPOCH 3
EPOCH 4

~~~~~~~~~~~~~~~~~
TRIAL: ConorNet.Hyperparams(k=7, activation='tanh', learning_rate=0.1, training_epochs=5, batch_size=5)
Dataset: two_gaussians; Device: cuda
EPOCH 0
EPOCH 1
EPOCH 2
EPOCH 3
EPOCH 4

~~~~~~~~~~~~~~~~~
TRIAL: ConorNet.Hyperparams(k=7, activation='tanh', learning_rate=0.1, training_epochs=5, batch_size=50)
Dataset: two_gaussians; Device: cuda
EPOCH 0
EPOCH 1
EPOCH 2
EPOCH 3
EPOCH 4

~~~~~~~~~~~~~~~~~
TRIAL: ConorNet.Hyperparams(k=7, activation='tanh', learning_rate=0.1, training_epochs=5, batch_size=100)
Dataset: two_gaussians; Device: cuda
EPOCH 0
EPOCH 1
EPOCH 2
EPOCH 3
EPOCH 4

~~~~~~~~~~~~~~~~~
TRIAL: ConorNet.Hyperparams(k=7, activation='tanh', learning_rate=0.1, training_epochs=10, batch_size=1)
Dataset: two_gaussians; Device: cuda
EPOCH 0
EPOCH 1
EPOCH 2
EPOCH 3
EPOC